# VIVS: Variable Importance via Variance Statistics

This tutorial demonstrates how to use VIVS to identify spatially variable genes in spatial transcriptomics data.

VIVS compares observed and expected variance at different spatial scales to identify genes with significant spatial patterns.

## Plan for this tutorial:
1. Loading spatial data
2. Setting up and fitting the VIVS model
3. Identifying spatially variable genes
4. Visualizing results

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

# Import spatialvi
import spatialvi
from spatialvi.external import VIVS

sc.set_figure_params(figsize=(6, 6))
print("spatialvi version:", spatialvi.__version__)

## 1. Load Data

Load a spatial transcriptomics dataset. Here we use an example Visium dataset.

In [ ]:
# Load example spatial data
# Replace with your own data path
adata = sc.datasets.visium_sge(sample_id="V1_Human_Lymph_Node")
adata.var_names_make_unique()

# Basic preprocessing
sc.pp.filter_genes(adata, min_cells=10)
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor="seurat_v3")

print(adata)

In [ ]:
# Visualize the spatial data
sc.pl.spatial(adata, color="total_counts", spot_size=100)

## 2. Set up VIVS Model

Initialize the VIVS model with the spatial data.

In [ ]:
# Initialize VIVS model
model = VIVS(
    adata,
    spatial_key="spatial",  # Key in adata.obsm for spatial coordinates
    n_neighbors=20,  # Number of spatial neighbors
    n_scales=5,  # Number of spatial scales to analyze
)

print("VIVS model initialized")

## 3. Fit the Model

Fit VIVS to identify spatially variable genes.

In [ ]:
# Fit the model
model.fit(
    n_permutations=100,  # Number of permutations for null distribution
    batch_size=100,  # Batch size for processing
    use_gpu=False,  # Set True if GPU available
)

print("Model fitting complete!")

## 4. Get Spatially Variable Genes

In [ ]:
# Get spatially variable genes
svg_df = model.get_spatially_variable_genes()

# Display top spatially variable genes
print("Top 20 spatially variable genes:")
svg_df.head(20)

In [ ]:
# Get importance scores
importance = model.get_importance_scores()
print(f"Importance scores shape: {importance.shape}")

## 5. Visualize Results

In [ ]:
# Visualize top spatially variable genes
top_genes = svg_df.head(6).index.tolist()

sc.pl.spatial(adata, color=top_genes, ncols=3, spot_size=80)

In [ ]:
# Plot importance score distribution
plt.figure(figsize=(8, 4))
plt.hist(svg_df["importance_score"], bins=50, edgecolor="black")
plt.xlabel("Importance Score")
plt.ylabel("Number of Genes")
plt.title("Distribution of Spatial Importance Scores")
plt.show()

## Multi-scale Analysis

VIVS can also perform multi-scale analysis to identify genes with spatial patterns at different scales.

In [ ]:
# Get scale-specific scores
scale_scores = model.get_scale_scores()
print(f"Scale scores shape: {scale_scores.shape}")

# Plot scale-specific patterns for top genes
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, gene in enumerate(top_genes[:3]):
    gene_idx = list(adata.var_names).index(gene)
    axes[i].plot(scale_scores[gene_idx], marker="o")
    axes[i].set_xlabel("Scale")
    axes[i].set_ylabel("Score")
    axes[i].set_title(f"{gene}")
plt.tight_layout()
plt.show()

## Summary

In this tutorial, we demonstrated:
1. How to load and preprocess spatial transcriptomics data
2. How to set up and fit the VIVS model
3. How to identify spatially variable genes
4. How to visualize the results

VIVS provides a powerful approach for identifying genes with significant spatial patterns without requiring prior knowledge of tissue structure.